# N066 · 切分回溯与字符串约束

**目标：** 把切分边界当作选择并维护合法前缀。

**先修：** N023, N064。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 切分树；回文判断；前导零；剩余长度剪枝；恢复。

## 从问题到算法

切分问题的状态是“已经处理到哪个下标”，每次选择下一段的结束位置。回文切分先预计算任意区间是否回文，避免同一段反复判断。恢复IP地址必须同时满足四段、每段0..255、无多余前导零，剩余字符长度还可用作剪枝。

## 最小实现

**本章接口：** `partition_palindromes(s)`、`restore_ip_addresses(s)`

In [1]:
def partition_palindromes(s):
    n=len(s); palindrome=[[False]*n for _ in range(n)]
    for left in range(n-1,-1,-1):
        for right in range(left,n):
            palindrome[left][right]=s[left]==s[right] and (right-left<2 or palindrome[left+1][right-1])
    out=[]; path=[]
    def visit(start):
        if start==n: out.append(path[:]); return
        for end in range(start,n):
            if palindrome[start][end]:
                path.append(s[start:end+1]); visit(end+1); path.pop()
    visit(0); return out

def restore_ip_addresses(s):
    if any(c not in '0123456789' for c in s): raise ValueError('ASCII decimal digits required')
    out=[]; path=[]
    def visit(start):
        needed=4-len(path); remaining=len(s)-start
        if not needed<=remaining<=3*needed: return
        if needed==0:
            out.append('.'.join(path)); return
        for length in range(1,4):
            if start+length>len(s): break
            part=s[start:start+length]
            if len(part)>1 and part[0]=='0': break
            if int(part)>255: break
            path.append(part); visit(start+length); path.pop()
    visit(0); return out

## 正确性、前提与复杂度

任意切分有唯一递增切点序列，枚举所有合法下一段即可完备。IP长度剪枝只删除不可能把剩余字符放入剩余1..3位段的状态。

**代价：** 回文预处理O(n²)，输出最坏指数；IP至多3⁴条深度4路径，输入长度超过12时可立即排除。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['切分','段长','全为回文'],[(p,list(map(len,p)),all(x==x[::-1] for x in p)) for p in partition_palindromes('aab')])
show_table(['输入','合法IP'],[('25525511135',restore_ip_addresses('25525511135')),('0000',restore_ip_addresses('0000'))])

切分,段长,全为回文
"['a', 'a', 'b']","[1, 1, 1]",True
"['aa', 'b']","[2, 1]",True


输入,合法IP
25525511135,"['255.255.11.135', '255.255.111.35']"
0000,['0.0.0.0']


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert sorted(partition_palindromes('aab'))==[['a','a','b'],['aa','b']]
assert set(restore_ip_addresses('25525511135'))=={'255.255.11.135','255.255.111.35'}
assert restore_ip_addresses('0000')==['0.0.0.0']
for s in ['','a','abba','aabaa']:
    for parts in partition_palindromes(s): assert ''.join(parts)==s and all(x==x[::-1] for x in parts)
from itertools import combinations
for s in ['101023','1111','00000','1234567']:
    ref=[]
    for cuts in combinations(range(1,len(s)),3):
        bounds=(0,*cuts,len(s)); parts=[s[bounds[i]:bounds[i+1]] for i in range(4)]
        if all(len(x)<=3 and (len(x)==1 or x[0]!='0') and int(x)<=255 for x in parts): ref.append('.'.join(parts))
    assert sorted(restore_ip_addresses(s))==sorted(ref)
print('N066: 所有本章断言通过')

N066: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 预计算回文表作为进阶对照。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明IP片段不能带多余前导零。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 131. Palindrome Partitioning（canonical）
- 93. Restore IP Addresses（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。